# 天気図パターン分類 - すぐに使う (v9)

1. まず下の「セットアップ」セルを1回実行してください(コードは折りたたまれています)。
2. その後、一番下の入力フォームで条件を選んで実行してください。

> セットアップセル実行後に表示される `バージョン: vX` が、上のタイトルの版と
> 一致していれば最新版が動いています。古い場合はランタイムを再接続し、
> ページを再読み込みしてください。

In [ ]:
#@title 🔧 セットアップ(最初に1回だけ実行してください) { display-mode: "form" }
NOTEBOOK_VERSION = "v9"

REPO_URL = "https://github.com/awg-yk/weather-pattern-classification.git"
# mainはコードのみ(天気図の画像データを含まない)ので数秒でcloneできる。
# 画像データを含むブランチはcloneすると3GB超になり事実上終わらないため使わない。
BRANCH = "main"
REPO_DIR = "/content/weather-pattern-classification"
WEIGHTS_PATH = f"{REPO_DIR}/weights/model.pt"

import subprocess, os, sys, shutil, time

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

os.environ["GIT_TERMINAL_PROMPT"] = "0"  # 認証プロンプト待ちで固まるのを防ぐ
print("リポジトリ取得中...")
t0 = time.time()
clone = subprocess.run(
    ["timeout", "120", "git", "clone", "--depth", "1", "--single-branch",
     "-b", BRANCH, REPO_URL, REPO_DIR],
    check=False,
)
if clone.returncode == 124:
    raise TimeoutError(
        "git cloneが120秒経っても終わりませんでした。\n"
        "ランタイムを再接続してから、もう一度実行してください。"
    )
elif clone.returncode != 0:
    raise RuntimeError(f"git cloneに失敗しました(終了コード: {clone.returncode})")
print(f"取得完了 ({time.time() - t0:.1f}秒)")

%cd {REPO_DIR}
# Colabにはtorch/torchvision/numpy/pandas/matplotlib/requests/tqdm/pillowが
# 標準搭載済みなので、追加で必要なpdf2imageだけ入れる。
!pip install -q pdf2image
!apt-get -qq install -y fonts-noto-cjk poppler-utils

assert os.path.exists(WEIGHTS_PATH), f"モデルの重みが見つかりません: {WEIGHTS_PATH}"

sys.path.append(REPO_DIR)
import matplotlib.pyplot as plt
from src.labels import LABEL_JA
from scripts.gradcam import explain_predictions_above_threshold
from scripts.fetch_and_predict import fetch_chart, chart_exists
from google.colab import files


def classify_and_show(image_path: str, threshold: float):
    """画像1枚を分類し、確信度がthresholdを超えたラベル分だけヒートマップを表示、
    それ以外はテキストのみで確信度一覧を出す。"""
    display_image, overlays, ranked = explain_predictions_above_threshold(
        image_path=image_path,
        weights_path=WEIGHTS_PATH,
        threshold=threshold,
        apply_preprocess=True,
    )

    n_panels = len(overlays) + 1
    fig, axes = plt.subplots(1, n_panels, figsize=(5 * n_panels, 5))
    if n_panels == 1:
        axes = [axes]
    axes[0].imshow(display_image)
    axes[0].set_title("入力画像(前処理後)")
    axes[0].axis("off")

    for ax, (label, prob, overlay) in zip(axes[1:], overlays):
        ax.imshow(overlay)
        ax.set_title(f"{LABEL_JA[label]}\n({prob * 100:.1f}%)")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

    if not overlays:
        print(f"確信度{threshold * 100:.0f}%を超えるラベルはありませんでした。\n")

    print("--- 全ラベルの確信度 ---")
    for label, prob in ranked:
        print(f"{LABEL_JA[label]}: {prob * 100:.1f}%")


print(f"セットアップ完了 | バージョン: {NOTEBOOK_VERSION}")

In [ ]:
#@title 画像を用意して分類 { display-mode: "form" }
#@markdown **MODE**: `upload`=画像をアップロード / `date`=日付指定で取得(2000-01-01〜2022-09-30は手動アーカイブ、2022-10-01以降は気象庁JSMAPアーカイブから自動取得)
MODE = "date" #@param ["upload", "date"]

#@markdown **日付**(MODE="date"のときのみ使用)
YEAR = 2025 #@param [2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026] {type:"raw"}
MONTH = 1 #@param [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12] {type:"raw"}
DAY = 1 #@param [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31] {type:"raw"}
HOUR = 0 #@param [0, 12] {type:"raw"}

#@markdown **表示するラベルの確信度しきい値**(これを超えたラベルだけヒートマップ画像で表示)
THRESHOLD = 0.5 #@param {type:"slider", min:0.0, max:1.0, step:0.05}

if MODE == "upload":
    uploaded = files.upload()
    image_path = list(uploaded.keys())[0]
elif MODE == "date":
    import datetime
    from scripts.fetch_and_predict import EARLIEST_KNOWN_DATE
    from scripts.fetch_manual_chart import MANUAL_ARCHIVE_START_DATE
    target = datetime.date(YEAR, MONTH, DAY)

    if not chart_exists(target, HOUR):
        raise FileNotFoundError(
            f"{target.isoformat()} {HOUR}Z の天気図が見つかりません。"
            f"対応範囲は{MANUAL_ARCHIVE_START_DATE.isoformat()}以降(0Zまたは12Zのみ)です。"
        )

    date_str = f"{YEAR:04d}-{MONTH:02d}-{DAY:02d}"
    image_path = str(fetch_chart(date_str, hour=HOUR))
    print("取得:", image_path)
else:
    raise ValueError('MODEは "upload" か "date" を指定してください')

classify_and_show(image_path, threshold=THRESHOLD)